# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aleeza-Maryam/Aleeza-ML-Internship-WEEK1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
import duckdb

con = duckdb.connect()

print("DuckDB connected")
con.execute(
    f"""
    CREATE SECRET (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

print("Hugging Face connection configured")
HF_DATASET = "hf://datasets/FlyRank/internship-warehouse"
con.sql(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{HF_DATASET}/fact_content_daily_performance/**/*.parquet'
    )
    LIMIT 1
    """
).show()

DuckDB connected
Hugging Face connection configured
┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ B

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Token loaded: True


## 1) My lane's data contract

**One row means:** One content item for one client on one report date.

**Table:** `fact_content_daily_performance`

**Time window:** I will use March 2026 (`month = '2026-03'`) as the development month. June 2026 is treated as a sealed final/outcome month.

**Prediction/ranking target:** Rank content items by their likelihood of declining or needing a content refresh.

**Deliberate exclusion:** I will exclude label-derived and decision-derived fields from the model features because they would leak information about the outcome.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2) Fields: feature / label / context / excluded

| Field | Category | Why |
|---|---|---|
| `gsc_impressions` | Feature | Observable search performance before the decision |
| `gsc_clicks` | Feature | Observable search traffic signal |
| `gsc_sum_position` | Feature | Observable search ranking signal |
| `report_date` | Context | Identifies when the observation was recorded |
| `month` | Context | Used to select the development time window |
| `client_hash_id` | Context | Identifies the client without exposing identity |
| `content_hash_id` | Context | Identifies the content item |
| Declining status | Label | The outcome we want to predict |
| `trend_direction` | Excluded | Derived from the outcome and can leak the label |
| Decision/recommendation fields | Excluded | They represent an existing decision rather than an input known before prediction |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token exists:", HF_TOKEN is not None)
print("Token starts correctly:", HF_TOKEN.startswith("hf_") if HF_TOKEN else False)
from huggingface_hub import whoami

info = whoami(token=HF_TOKEN)

print("Hugging Face authentication successful.")
print("User:", info["name"])
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

print("Dataset accessible.")
print("Number of files:", len(files))
print(files[:10])
con.execute("DROP SECRET IF EXISTS secret")
con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("DuckDB Hugging Face authentication configured.")


Token exists: True
Token starts correctly: True
Hugging Face authentication successful.
User: Aleeza50
Dataset accessible.
Number of files: 24
['.gitattributes', 'README.md', 'dim_clients.parquet', 'dim_content.parquet', 'fact_content_daily_performance/month=2025-01/data_0.parquet', 'fact_content_daily_performance/month=2025-02/data_0.parquet', 'fact_content_daily_performance/month=2025-03/data_0.parquet', 'fact_content_daily_performance/month=2025-04/data_0.parquet', 'fact_content_daily_performance/month=2025-05/data_0.parquet', 'fact_content_daily_performance/month=2025-06/data_0.parquet']


InvalidInputException: Invalid Input Error: Temporary secret with name 'hf_secret' already exists!

In [25]:
q1 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT
        client_hash_id || '|' ||
        content_hash_id || '|' ||
        CAST(report_date AS VARCHAR)
    ) AS distinct_grain_rows
FROM read_parquet(
    '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
""")

q1.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────┐
│ total_rows │ distinct_grain_rows │
│   int64    │        int64        │
├────────────┼─────────────────────┤
│    9841378 │             9841378 │
└────────────┴─────────────────────┘



In [26]:
q2 = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
""")

q2.show()

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘



In [27]:
q3 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows
FROM read_parquet(
    '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
""")

q3.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │
│   int64    │       int64        │
├────────────┼────────────────────┤
│    9841378 │            3611061 │
└────────────┴────────────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Token exists: True
Token starts correctly: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.